In [10]:
"""
City-level Baseline vs Cooperative Analysis (Heat side and Electric side)
Notes:
- This script computes simple cooperative benefits without storage (hourly sharing).
- It outputs city-level KPIs and simple economics (annual cost, NPV, payback).
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [12]:
# ---------- Economic parameters ----------
YEARS = 25
PRICE_GROWTH = 0.02
DISCOUNT_RATE = 0.04
CITY_CAPEX = 1e6
ANNUAL_OPEX = 10000
ELECTRIC_BUY_PRICE = 0.25
ELECTRIC_FEEDIN_PRICE = 0.05

# ---------- Data paths ----------
CITY_PATHS = {
    "KL": r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent",
    "PS": r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\PS_TS\CEAAgent",
}



In [17]:
# ==========================================================
# Simulation helpers
# ==========================================================
def simulate_baseline(buildings: list[pd.DataFrame]) -> tuple[list[dict], pd.DataFrame]:
    """Baseline: each building consumes its own generation without sharing."""
    hourly = []
    results = []

    for b_id, df in enumerate(buildings):
        gen = np.maximum(np.nan_to_num(df["column30"].values, nan=0.0), 0.0)
        load = np.maximum(np.nan_to_num(df["column3"].values, nan=0.0), 0.0)

        self_use = np.minimum(gen, load)
        surplus = np.maximum(gen - load, 0.0)
        deficit = np.maximum(load - gen, 0.0)

        results.append({
            "self_use": float(self_use.sum()),
            "surplus": float(surplus.sum()),
            "deficit": float(deficit.sum()),
            "total_gen": float(gen.sum()),
            "total_load": float(load.sum()),
            "peak_grid": float(deficit.max()),
        })

        hourly.append(pd.DataFrame({
            "hour": np.arange(len(gen)),
            "building": b_id,
            "gen": gen,
            "load": load,
            "self_use": self_use,
            "surplus": surplus,
            "deficit": deficit,
        }))

    hourly_df = pd.concat(hourly, ignore_index=True)
    return results, hourly_df


def simulate_coop(buildings: list[pd.DataFrame]) -> tuple[dict, pd.DataFrame]:
    """Cooperative sharing among all members at each hour."""
    T = len(buildings[0]["column3"])
    hourly = []

    coop_self_use, coop_surplus_wasted, coop_deficit_after_share = 0.0, 0.0, 0.0
    peak_grid_purchase = 0.0

    for t in range(T):
        gen_t = np.array([max(0.0, b["column30"].iat[t]) for b in buildings], dtype=float)
        load_t = np.array([max(0.0, b["column3"].iat[t]) for b in buildings], dtype=float)

        self_use_t = np.minimum(gen_t, load_t)
        surplus_t = np.maximum(gen_t - load_t, 0.0)
        deficit_t = np.maximum(load_t - gen_t, 0.0)

        total_surplus = float(surplus_t.sum())
        total_deficit = float(deficit_t.sum())
        shared_used = min(total_surplus, total_deficit)

        coop_self_use += float(self_use_t.sum()) + shared_used
        coop_surplus_wasted += total_surplus - shared_used
        coop_deficit_after_share += total_deficit - shared_used
        peak_grid_purchase = max(peak_grid_purchase, total_deficit - shared_used)

        hourly.append({
            "hour": t,
            "gen_total": gen_t.sum(),
            "load_total": load_t.sum(),
            "self_use": self_use_t.sum(),
            "surplus_total": total_surplus,
            "deficit_total": total_deficit,
            "shared_used": shared_used,
            "deficit_after_share": total_deficit - shared_used,
        })

    total_gen = float(sum([np.maximum(0.0, b["column30"]).sum() for b in buildings]))
    total_load = float(sum([np.maximum(0.0, b["column3"]).sum() for b in buildings]))

    annual = {
        "self_use": coop_self_use,
        "surplus_wasted": coop_surplus_wasted,
        "deficit_after_share": coop_deficit_after_share,
        "total_gen": total_gen,
        "total_load": total_load,
        "peak_grid": peak_grid_purchase,
    }

    return annual, pd.DataFrame(hourly)


def calc_kpis(total_gen, total_load, self_use, surplus_wasted, peak_grid) -> dict:
    """Compute key performance indicators."""
    sc_ratio = self_use / total_gen if total_gen > 0 else 0.0
    ss_ratio = self_use / total_load if total_load > 0 else 0.0
    curtail = surplus_wasted / total_gen if total_gen > 0 else 0.0
    return {
        "self_consumption": sc_ratio,
        "self_sufficiency": ss_ratio,
        "curtailment": curtail,
        "peak_grid": peak_grid,
    }


# ==========================================================
# Economics
# ==========================================================
def gas_cost(h_kwh: float) -> float:
    """Annual gas cost function."""
    h = h_kwh / 0.8
    if h <= 2000:
        return 6.00 * 12 + (20.47 + 2.226) * h / 100
    elif h <= 10000:
        return 8.00 * 12 + (18.62 + 2.226) * h / 100
    elif h <= 30000:
        return 12.00 * 12 + (17.73 + 2.226) * h / 100
    elif h <= 100000:
        return 14.00 * 12 + (17.61 + 2.226) * h / 100
    elif h <= 300000:
        return 24.00 * 12 + (17.37 + 2.226) * h / 100
    else:
        return 60.00 * 12 + (17.2 + 2.226) * h / 100


def annual_cost_heat(deficit_kwh: float) -> float:
    return float(gas_cost(deficit_kwh))


def annual_cost_electric(deficit_kwh: float, surplus_wasted: float,
                         buy_price: float, feedin_price: float) -> float:
    purchase_cost = deficit_kwh * buy_price
    feedin_revenue = surplus_wasted * feedin_price
    return float(purchase_cost - feedin_revenue)


# ==========================================================
# NPV & Payback
# ==========================================================
def npv_from_annual_saving(annual_saving: float,
                           years: int = YEARS,
                           growth: float = PRICE_GROWTH,
                           discount: float = DISCOUNT_RATE,
                           capex: float = CITY_CAPEX,
                           opex: float = ANNUAL_OPEX) -> tuple[float, list[float]]:
    """Compute NPV and cashflows over project lifetime."""
    cashflows = [-capex]
    npv = -capex
    for y in range(1, years + 1):
        benefit_y = annual_saving * ((1 + growth) ** y) - opex
        disc = benefit_y / ((1 + discount) ** y)
        cashflows.append(disc)
        npv += disc
    return npv, cashflows


def payback_year(cashflows: list[float]) -> float | None:
    """Return the first year when cumulative cashflow >= 0."""
    cum = 0.0
    for y, cf in enumerate(cashflows):
        cum += cf
        if cum >= 0:
            return float(y)
    return None


# ==========================================================
# City analysis
# ==========================================================
def load_city_buildings(data_dir: str) -> list[pd.DataFrame]:
    """Load building CSVs and print progress every 10 files."""
    buildings = []
    count = 0
    files = [fn for fn in os.listdir(data_dir) if fn.endswith(".csv")]
    total = len(files)

    for fn in files:
        df = pd.read_csv(os.path.join(data_dir, fn))
        if {"column3", "column30"}.issubset(df.columns):
            buildings.append(df)
        count += 1
        if count % 10 == 0:
            print(f"   🔄 Processed {count}/{total} files in {data_dir}...")
    print(f"   ✅ Finished loading {count} files from {data_dir}")
    return buildings


def analyze_city(city: str, data_dir: str, mode: str):
    print(f"\n▶️ Analyzing {city} - {mode} ...")
    buildings = load_city_buildings(data_dir)
    if not buildings:
        print(f"⚠️ No valid data for {city} - {mode}")
        return None

    # Baseline
    base_res, hourly_base = simulate_baseline(buildings)
    gen_b, load_b = sum(r["total_gen"] for r in base_res), sum(r["total_load"] for r in base_res)
    selfuse_b, surplus_b, deficit_b = sum(r["self_use"] for r in base_res), sum(r["surplus"] for r in base_res), sum(r["deficit"] for r in base_res)
    peak_b = max(r["peak_grid"] for r in base_res)
    kpis_b = calc_kpis(gen_b, load_b, selfuse_b, surplus_b, peak_b)
    cost_b = annual_cost_heat(deficit_b) if mode == "heat" else annual_cost_electric(deficit_b, surplus_b, ELECTRIC_BUY_PRICE, ELECTRIC_FEEDIN_PRICE)

    # Cooperative
    coop_res, hourly_coop = simulate_coop(buildings)
    kpis_c = calc_kpis(coop_res["total_gen"], coop_res["total_load"], coop_res["self_use"], coop_res["surplus_wasted"], coop_res["peak_grid"])
    cost_c = annual_cost_heat(coop_res["deficit_after_share"]) if mode == "heat" else annual_cost_electric(coop_res["deficit_after_share"], coop_res["surplus_wasted"], ELECTRIC_BUY_PRICE, ELECTRIC_FEEDIN_PRICE)

    saving_c = cost_b - cost_c
    npv_c, flows_c = npv_from_annual_saving(saving_c)
    payback_c = payback_year(flows_c)

    # Save hourly
    hourly_base.to_csv(f"{city}_{mode}_hourly_baseline.csv", index=False)
    hourly_coop.to_csv(f"{city}_{mode}_hourly_coop.csv", index=False)
    print(f"   ⏺️ Saved hourly CSVs for {city} - {mode}")

    # Annual summary
    df = pd.DataFrame([
        {"city": city, "mode": mode, "scenario": "baseline", **kpis_b,
         "annual_cost": cost_b, "annual_saving": 0.0, "npv": 0.0, "payback_year": None},
        {"city": city, "mode": mode, "scenario": "cooperative", **kpis_c,
         "annual_cost": cost_c, "annual_saving": saving_c, "npv": npv_c, "payback_year": payback_c},
    ])
    print(f"   ✅ Finished {city} - {mode}")
    return df


# ==========================================================
# Cross-city analysis
# ==========================================================
def analyze_cross_city(city_pair: list[str], paths: dict, mode: str):
    print(f"\n▶️ Cross-city analysis: {'+'.join(city_pair)} - {mode}")
    all_buildings = []
    baseline_self, baseline_surplus, baseline_deficit = 0.0, 0.0, 0.0
    baseline_gen, baseline_load, baseline_peak = 0.0, 0.0, 0.0

    for city in city_pair:
        buildings = load_city_buildings(paths[city])
        if not buildings: continue
        all_buildings.extend(buildings)
        coop_res, _ = simulate_coop(buildings)
        baseline_gen += coop_res["total_gen"]
        baseline_load += coop_res["total_load"]
        baseline_self += coop_res["self_use"]
        baseline_surplus += coop_res["surplus_wasted"]
        baseline_deficit += coop_res["deficit_after_share"]
        baseline_peak = max(baseline_peak, coop_res["peak_grid"])

    kpis_b = calc_kpis(baseline_gen, baseline_load, baseline_self, baseline_surplus, baseline_peak)

    # Joint cooperative
    coop_res, hourly_coop = simulate_coop(all_buildings)
    kpis_c = calc_kpis(coop_res["total_gen"], coop_res["total_load"], coop_res["self_use"], coop_res["surplus_wasted"], coop_res["peak_grid"])

    hourly_coop.to_csv(f"{'+'.join(city_pair)}_{mode}_hourly_coop.csv", index=False)
    print(f"   ⏺️ Saved hourly CSV for {'+'.join(city_pair)} - {mode}")
    print(f"   ✅ Finished cross-city {mode}")

    return pd.DataFrame([
        {"city": "+".join(city_pair), "mode": mode, "scenario": "baseline", **kpis_b},
        {"city": "+".join(city_pair), "mode": mode, "scenario": "cooperative", **kpis_c},
    ])

In [ ]:

# ==========================================================
# Main
# ==========================================================
if __name__ == "__main__":
    results = []

    # Per city
    for city, path in CITY_PATHS.items():
        for mode in ["heat", "electric"]:
            df = analyze_city(city, path, mode)
            if df is not None:
                df.to_csv(f"{city}_{mode}_annual.csv", index=False)
                results.append(df)

    # Save combined
    if results:
        pd.concat(results, ignore_index=True).to_csv("all_city_annual.csv", index=False)

    # Cross city (KL + PS)
    for mode in ["heat", "electric"]:
        df_cross = analyze_cross_city(["KL", "PS"], CITY_PATHS, mode)
        if df_cross is not None:
            df_cross.to_csv(f"KL+PS_{mode}_annual.csv", index=False)



▶️ Analyzing KL - heat ...
   🔄 Processed 10/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 20/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 30/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 40/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 50/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 60/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 70/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 80/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 90/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 100/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\